# Week 01 · Problem framing and baselines · Lab

Today we take a vague request and turn it into a question a model can actually answer. Then, before training anything serious, we measure how well we can do by barely trying.

**The question:** at the moment a customer places an order, can we predict whether it will arrive after the date we promised?

**How this lab works**

- Cells marked **TODO** are yours to complete.
- Cells marked **Check** test your answer. If a check fails, the message tells you what to look at.
- Work in your own copy, not in this file: *File → Save As* and call it `my-lab.ipynb`. Then `git pull` next week can never clash with your changes.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, average_precision_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

RANDOM_STATE = 42
DATA = Path("..") / "data" / "raw" / "olist"

assert DATA.exists(), "No data found. From the repository root, run: python data/download.py"
pd.set_option("display.max_columns", 20)

## 1. Meet the data

Olist's data comes as nine tables. Today we only need two of them:

- **orders**: one row per order, with its status and five timestamps
- **customers**: who placed each order, and where they live

We tell pandas which columns hold dates, so it reads them as dates rather than as text.

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders = pd.read_csv(DATA / "olist_orders_dataset.csv", parse_dates=date_columns)
customers = pd.read_csv(DATA / "olist_customers_dataset.csv")

print("orders:   ", orders.shape)
print("customers:", customers.shape)
orders.head()

The five timestamps tell the story of one order, in this order:

| Column | When it is recorded |
|---|---|
| `order_purchase_timestamp` | The customer clicks buy |
| `order_approved_at` | The payment is approved |
| `order_delivered_carrier_date` | The seller hands the parcel to the carrier |
| `order_delivered_customer_date` | The parcel reaches the customer |
| `order_estimated_delivery_date` | The date promised to the customer at checkout |

Keep this table in mind. It becomes very important in Part 4.

The first two questions to ask of any new table: what type did each column get, and what is missing?

In [ ]:
orders.info()

In [ ]:
# TODO: count the missing values in each column of `orders`

Some orders have no delivery date. The status column explains why:

In [ ]:
orders["order_status"].value_counts()

Most orders were delivered. The rest were cancelled, never shipped, or were still on their way when the data was collected.

One more thing before moving on. A row in `customers` is not a person, it is an order: `customer_id` is created fresh for every purchase. The actual person is `customer_unique_id`.

In [ ]:
print("customer rows:    ", len(customers))
print("different people: ", customers["customer_unique_id"].nunique())

About three thousand people ordered more than once. It does not matter today. Remember it anyway, because it matters a lot in Week 2.

## 2. Define the target

"Late" sounds obvious. It is not. Before we can predict it, we have to decide exactly what it means, and every decision changes the data.

### Which orders can we label?

An order that never arrived has no delivery date, so we cannot say whether it was late. We leave those orders out. That is a real decision and it belongs in writing: our model will only ever learn about orders that were delivered.

In [ ]:
delivered = orders[orders["order_delivered_customer_date"].notna()].copy()
print(f"kept {len(delivered):,} of {len(orders):,} orders")

### Which months can we trust?

In [ ]:
per_month = delivered["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index()
per_month

The start is thin: one order in September 2016, one in December, and a short burst in October with a gap after it. The business only really gets going in January 2017. At the other end, August 2018 is the last full month.

We keep **January 2017 to August 2018**: twenty months.

In [ ]:
START, END = "2017-01-01", "2018-09-01"

in_window = delivered["order_purchase_timestamp"].between(START, END, inclusive="left")
delivered = delivered[in_window].copy()
print(f"{len(delivered):,} orders from January 2017 to August 2018")

### What exactly counts as late?

The obvious rule: an order is late if it was delivered after the promised date. Before using it, look closely at the promised date itself.

In [ ]:
delivered["order_estimated_delivery_date"].dt.time.value_counts()

Every single promised date is at midnight. The promise is a **day**, with no time attached. The delivery, on the other hand, has a full timestamp.

So what happens to a parcel delivered at two in the afternoon on the promised day? Let's compare two ways of defining late.

In [ ]:
promised = delivered["order_estimated_delivery_date"]
arrived = delivered["order_delivered_customer_date"]

# TODO: late_by_timestamp compares the two timestamps directly
late_by_timestamp = ...

# TODO: late_by_day compares calendar days only
#       Hint: .dt.normalize() drops the time of day and keeps the date
late_by_day = ...

print(f"late by timestamp:    {late_by_timestamp.mean():.1%}")
print(f"late by calendar day: {late_by_day.mean():.1%}")
print(f"orders that differ:   {(late_by_timestamp != late_by_day).sum():,}")

In [ ]:
# Check
assert abs(late_by_timestamp.mean() - 0.081) < 0.002, "late_by_timestamp should be around 8.1%. Compare `arrived > promised` directly."
assert abs(late_by_day.mean() - 0.068) < 0.002, "late_by_day should be around 6.8%. Did you normalize the arrival time before comparing?"
print("Correct.")

About 1,300 orders arrived **on** the promised day, and the timestamp rule calls every one of them late. A customer promised Tuesday who receives the parcel on Tuesday afternoon would not call that late, so neither do we.

This is the point of the section. **The target is something you decide, not something you find.** One small choice moved the late rate from 8.1% to 6.8%, and the wrong choice would have silently mislabelled over a thousand orders.

In [ ]:
delivered["is_late"] = late_by_day.astype(int)
delivered["is_late"].value_counts(normalize=True)

About one order in fifteen is late. Keep that in mind: a model that always answers "on time" will be right about 93% of the time without learning anything at all.

## 3. Look before you model

First we add each customer's state to their order. Then we look at how the late rate moves over time, and across the country.

In [ ]:
df = delivered.merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")

print(len(delivered), len(df))    # always check: the join should not add or lose rows

### Over time

In [ ]:
monthly = df.groupby(df["order_purchase_timestamp"].dt.to_period("M"))["is_late"].mean()

ax = monthly.plot(marker="o", figsize=(9, 3.5))
ax.set_title("Share of orders delivered late, by month of purchase")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(PercentFormatter(1))
plt.show()

This is not a flat line. For most of 2017 the late rate stays between about 3% and 5%. Then it jumps to about 12% in November 2017, the month of Black Friday, climbs to nearly 19% in March 2018, and then drops to about 1% in June.

Hold on to this picture. If the late rate changes this much from one month to the next, what does that mean for a model trained on one period and then used in another? That question opens Week 2.

### Across the country

Before you look, make a guess. Brazil is enormous, and the northern states are a long way from the big sellers in São Paulo. **Which states do you expect to have the most late deliveries?**

Write your guess here before running the next cell:

*My guess:* ...

In [ ]:
# TODO: one row per state, with two columns:
#       `orders`    how many orders went to that state
#       `late_rate` the share of them that were late
#       Sort from worst to best.
#       Hint: groupby("customer_state").agg(orders=(..., "size"), late_rate=(..., "mean"))
by_state = ...
by_state

In [ ]:
# Check
assert by_state.index[0] == "AL", "The worst state should come first. Did you sort by late_rate, descending?"
assert by_state.loc["SP", "orders"] > 30000, "`orders` should count the orders per state."
print("Correct.")

In [ ]:
ax = by_state["late_rate"].sort_values().plot.barh(figsize=(7, 7))
ax.set_title("Late rate by customer state")
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set_ylabel("")
plt.show()

If you guessed the far north, the data disagrees. Amazonas, Amapá and Rondônia are among the **best** states. Alagoas and Maranhão, in the northeast, are the worst, and Rio de Janeiro, right next door to São Paulo, is late about one time in eight.

A likely reason: customers far away are promised long delivery windows, so even a slow parcel arrives "on time". Lateness is measured against the promise, not against the distance.

The lesson is bigger than Brazil. **Check your intuition against the data before you build on it.** A rule based on our first guess would have flagged exactly the wrong states.

One more detail: look at the `orders` column. Roraima (RR) has only 40 orders, so two or three late parcels move its rate a lot. Rates computed from small groups are unreliable.

## 4. What do we know at the moment of prediction?

We want to predict lateness at the moment the customer clicks buy. So we may only use information that exists at that moment.

```
purchase ──► approved ──► handed to carrier ──► delivered
    ▲
    we predict here. Everything to the right is still in the future.
```

Back to the timestamp table from Part 1:

| Column | Known at purchase? |
|---|---|
| `order_purchase_timestamp` | Yes |
| `order_estimated_delivery_date` | Yes, the customer sees it at checkout |
| `customer_state` | Yes |
| `order_approved_at` | No, it comes later |
| `order_delivered_carrier_date` | No |
| `order_delivered_customer_date` | No, and it is the answer itself |

From the columns we are allowed to use, we build four simple features:

- `window_days`: how many days we promised the customer
- `weekday` and `hour`: when the order was placed
- `customer_state`: where it is going

In [ ]:
purchase_day = df["order_purchase_timestamp"].dt.normalize()

# TODO: window_days is the number of days between the purchase day and the promised date
#       Hint: subtract two date columns, then take .dt.days
df["window_days"] = ...

df["weekday"] = df["order_purchase_timestamp"].dt.dayofweek     # Monday is 0
df["hour"] = df["order_purchase_timestamp"].dt.hour

df[["window_days", "weekday", "hour"]].describe().round(1)

In [ ]:
# Check
assert df["window_days"].median() == 24, "The median promise should be 24 days. Subtract purchase_day from the promised date."
print("Correct.")

The typical promise is about three and a half weeks. Some customers are promised a few days, a handful more than three months.

## 5. Split the data, for now

We hold back 20% of the orders as a test set, and we do not look at how they turned out until we score our baselines on them. `stratify` keeps the late rate the same in both parts.

In [ ]:
train, test = train_test_split(df, test_size=0.2, stratify=df["is_late"], random_state=RANDOM_STATE)

print(f"train: {len(train):,} orders, {train['is_late'].mean():.1%} late")
print(f"test:  {len(test):,} orders, {test['is_late'].mean():.1%} late")

> **A warning we will come back to.** This is a random split, so test orders are scattered across all twenty months. Remember the monthly chart. Is that a fair test for a model that will be used on *next* month's orders? Week 2 answers this question, and the answer changes every number below.

## 6. The baseline ladder

Before any serious model, we climb three rungs. Each rung is the bar the next one has to clear.

We measure every rung in four ways:

- **accuracy**: the share of predictions that were right
- **precision**: of the orders we flagged as late, how many really were
- **recall**: of the orders that really were late, how many we flagged
- **PR AUC**: how well the model ranks late orders above on-time ones. Random guessing scores the late rate itself, about **0.068**, so that is the number to beat.

This small helper collects the results in one table as we go.

In [ ]:
results = []

def score(name, y_true, y_pred, y_score):
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "PR AUC": average_precision_score(y_true, y_score),
    })
    return pd.DataFrame(results).set_index("model").round(3)

### Rung 1: do nothing

A "model" that always predicts the most common answer, which is *on time*.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(train[["window_days"]], train["is_late"])

score(
    "always on time",
    test["is_late"],
    dummy.predict(test[["window_days"]]),
    dummy.predict_proba(test[["window_days"]])[:, 1],
)

Over 93% accuracy, and it has not caught a single late order.

**This is why accuracy is the wrong measure here.** When late orders are rare, doing nothing looks excellent. From now on, watch recall and PR AUC.

### Rung 2: a rule

A rule someone could write on a sticky note: *flag every order going to a state that is often late.*

We decide which states count as "often late" using the **training data only**. A state is high-risk if its late rate is at least one and a half times the average.

In [ ]:
state_rate = train.groupby("customer_state")["is_late"].mean()
average = train["is_late"].mean()

# TODO: high_risk holds the states whose late rate is at least 1.5 times the average
#       Hint: filter state_rate, then take .index
high_risk = ...
print("high-risk states:", sorted(high_risk))

# TODO: rule_pred is 1 for test orders going to a high-risk state, 0 otherwise
#       Hint: .isin(high_risk), then .astype(int)
rule_pred = ...

rule_score = test["customer_state"].map(state_rate).fillna(average)

score("rule: high-risk states", test["is_late"], rule_pred, rule_score)

In [ ]:
# Check
assert "AL" in high_risk and "SP" not in high_risk, "Alagoas should be high-risk and São Paulo should not."
assert abs(rule_pred.mean() - 0.23) < 0.03, "Your rule should flag roughly a quarter of test orders."
print("Correct.")

Accuracy dropped, and that is fine: the rule is finally flagging orders. It catches about four in ten late orders (recall), but most of its flags are false alarms (precision around 0.12). PR AUC moves from 0.068 to about 0.10. A small step, but a real one.

About `fillna(average)`: a state that never appears in the training data has no rate of its own, so it gets the average.

### Rung 3: a tiny model

A decision tree only two levels deep, so it can ask at most two questions about an order. We give it our four purchase-time features.

The tree needs numbers, not state names, so we replace each state with its late rate from the training data, the same numbers the rule used. Week 3 looks at this trick much more carefully.

Because late orders are rare, a plain tree would simply learn to answer "on time" every time. `class_weight="balanced"` tells it that the rare late orders matter as much as all the easy on-time ones.

In [ ]:
features = ["window_days", "weekday", "hour", "state_rate"]

def add_state_rate(frame):
    return frame.assign(state_rate=frame["customer_state"].map(state_rate).fillna(average))

X_train = add_state_rate(train)[features]
X_test = add_state_rate(test)[features]

# TODO: create a DecisionTreeClassifier with max_depth=2, class_weight="balanced"
#       and random_state=RANDOM_STATE, then fit it on X_train and train["is_late"]
tree = ...

score("depth-2 tree", test["is_late"], tree.predict(X_test), tree.predict_proba(X_test)[:, 1])

In [ ]:
# Check
assert tree.get_depth() == 2, "The tree should be exactly two levels deep."
assert results[-1]["recall"] > 0.4, "Recall should be above 0.4. Did you set class_weight='balanced'?"
print("Correct.")

Here is everything the tree learned, written as rules:

In [ ]:
print(export_text(tree, feature_names=features))

Read it from the top. In a state that is rarely late, only very short promises get flagged, under about ten days. In a state that is often late, almost any promise under a month gets flagged. Both rules make sense: the tree has found the same two ideas we saw in the charts.

The tree catches a few more late orders than the rule, with slightly better precision, but its PR AUC is about the same.

That is an honest result, and a useful one. With only these two tables and only what is known at purchase, there is not much signal to find. It tells us where to look next: sellers, products, weights and freight costs live in the other seven tables.

## 7. The trap

Imagine a teammate adds one more feature: how many days the delivery actually took.

**Before you run the next cell**, predict what will happen to precision, recall and PR AUC, and why:

*My prediction:* ...

In [ ]:
def add_leak(frame):
    took = frame["order_delivered_customer_date"] - frame["order_purchase_timestamp"]
    return add_state_rate(frame).assign(days_to_deliver=took.dt.days)

leaky_features = features + ["days_to_deliver"]

leaky_tree = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
leaky_tree.fit(add_leak(train)[leaky_features], train["is_late"])

X_test_leaky = add_leak(test)[leaky_features]
score(
    "LEAK: + actual delivery days",
    test["is_late"],
    leaky_tree.predict(X_test_leaky),
    leaky_tree.predict_proba(X_test_leaky)[:, 1],
)

Precision close to 0.96 and PR AUC close to 0.90. By far our best model, and completely useless.

The delivery time only exists once the parcel has arrived. At the moment of purchase, the moment we promised to predict at, this column does not exist yet. The model has not learned to predict lateness. It has learned to read the answer.

Notice what did **not** happen: nothing crashed, and no warning appeared. The score simply got better, which is exactly what makes leaks dangerous. Whenever a score jumps suspiciously, ask first: **could any feature only be known after the moment of prediction?**

This one was easy to spot. Week 3 hunts for much sneakier ones.

## 8. The problem card

Everything we decided today, in one place. Every project in this course starts with one of these.

**TODO:** fill in the right-hand column. Double-click this cell to edit it.

| | |
|---|---|
| **Question** | ... |
| **Target** | ... |
| **One row is** | ... |
| **Predict at** | ... |
| **Allowed information** | ... |
| **Left out** | ... |
| **Positive rate** | ... |
| **Decision it supports** | ... |
| **Costly mistake** | ... |
| **Metric** | ... |
| **Baseline to beat** | ... |

## 9. Ready for next week

Week 2 starts from the table we built today. Instead of saving a file, the course repository has a function that rebuilds it from the raw data in a few seconds. Let's confirm it gives exactly what we built here.

In [ ]:
import sys
sys.path.append("..")
from checkpoints import week_01

check = week_01()
merged = check.merge(df[["order_id", "is_late"]], on="order_id", suffixes=("", "_ours"))

print("same orders:", len(merged) == len(check) == len(df))
print("same labels:", (merged["is_late"] == merged["is_late_ours"]).all())

## What we learned

- **A vague request becomes a question** once you fix the target, what one row is, and the moment of prediction.
- **The target is a decision.** One small detail changed about 1,300 labels.
- **Look at the data before trusting your intuition.** The far north was not where the problem was.
- **Accuracy lies when one class is rare.** Doing nothing scored over 93%.
- **Our honest baselines barely beat nothing.** That is useful information, not a failure.
- **A leak does not crash anything.** It just makes the score look wonderful.

**Something to think about before next week:** our test orders were picked at random from all twenty months. What would happen if we tested the way the model will really be used, trained on the past and tested on the future?